# Scrape Reviews & Local News — 100 US Retail Stores

This notebook collects **customer reviews** and **local news** for **100 known-brand retail stores** across the USA, using **multiple public sources**.

It is separate from the 3–5 store demos in `02_real_store_scraper.ipynb` and `03_scrape_to_synthetic_format.ipynb`.

---

## Data sources

| Type | Sources | API key? |
|------|---------|----------|
| **Reviews** | Sitejabber, Trustpilot (brand pages), Google News RSS (review roundups) | No |
| **Local news** | Google News RSS, Bing News RSS (city + retailer + retail keywords) | No |
| **Optional downstream enrichment** | Azure OpenAI keys already present in `.env` | Yes, already available |

---

## Outputs

| File | Description |
|------|-------------|
| `data/scraped_100/customer_reviews.csv` | Review rows matching synthetic schema |
| `data/scraped_100/local_news.csv` | News rows matching synthetic schema |
| `data/scraped_100/scrape_run_log.csv` | Per-URL status log |
| `data/excel/scraped_100_reviews_news.xlsx` | Excel export for StoreDNA pipelines |

---

## Store catalog

- **100 stores** in `data/us_retail_stores_100.csv`
- **20 known US retailers** (Walmart, Target, Kroger, Costco, Publix, CVS, Home Depot, …)
- Spread across **50 major US cities**
- URL inventory: `data/us_retail_stores_100_urls.csv` (~630 review + news URLs)

> **Note:** This notebook does **not** require Google API keys. Review collection uses public review pages plus review-related RSS/news coverage. The Azure OpenAI keys already present in `.env` can be used later for enrichment or summarization, but they are not required for scraping itself.

## 1. Setup

In [1]:
%pip install -q -r ../requirements.txt

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
import os
import sys
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from src.reviews_news_collector import (
    CollectorConfig,
    build_us_retail_store_catalog,
    build_url_inventory,
    collect_all_stores,
)

load_dotenv(PROJECT_ROOT / ".env")

OUTPUT_DIR = PROJECT_ROOT / "data" / "scraped_100"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)

Project root: d:\AICOE\Retail-StoreDNA


## 2. Load 100-store catalog

In [8]:
CATALOG_PATH = PROJECT_ROOT / "data" / "us_retail_stores_100.csv"

if not CATALOG_PATH.exists():
    stores = build_us_retail_store_catalog(100)
    stores.to_csv(CATALOG_PATH, index=False)
else:
    stores = pd.read_csv(CATALOG_PATH)

print(f"Stores loaded: {len(stores)}")
stores.head(10)

Stores loaded: 100


,store_id,retailer,banner,store_name,city,state,store_format
0,USR-001,Walmart,Walmart,Walmart New York,New York,NY,Supercenter
1,USR-002,Target,Target,Target Los Angeles,Los Angeles,CA,Neighborhood
2,USR-003,Kroger,Kroger,Kroger Chicago,Chicago,IL,Neighborhood
3,USR-004,Costco,Costco,Costco Houston,Houston,TX,Supercenter
4,USR-005,Albertsons,Albertsons,Albertsons Phoenix,Phoenix,AZ,Neighborhood
5,USR-006,Publix,Publix,Publix Philadelphia,Philadelphia,PA,Neighborhood
6,USR-007,Whole Foods,Whole Foods,Whole Foods San Antonio,San Antonio,TX,Neighborhood
7,USR-008,CVS,CVS,CVS San Diego,San Diego,CA,Neighborhood
8,USR-009,Walgreens,Walgreens,Walgreens Dallas,Dallas,TX,Neighborhood
9,USR-010,Home Depot,Home Depot,Home Depot Austin,Austin,TX,Neighborhood


In [9]:
stores["retailer"].value_counts()

retailer
Walmart           5
Target            5
Kroger            5
Costco            5
Albertsons        5
Publix            5
Whole Foods       5
CVS               5
Walgreens         5
Home Depot        5
Lowe's            5
Best Buy          5
Dollar General    5
Sam's Club        5
Aldi              5
H-E-B             5
Meijer            5
Safeway           5
Sprouts           5
Trader Joe's      5
Name: count, dtype: int64

## 3. Configure sources

Set `MAX_STORES` below for a quick test (e.g. `5`) before running all 100.

This notebook uses only public review/news sources for scraping. No Google API key is needed.

In [16]:
MAX_STORES = 100           # set to 5 for a quick test run
SLEEP_SECONDS = 1.2       # delay between HTTP requests (be polite)
MAX_REVIEWS_PER_SOURCE = 8
MAX_NEWS_PER_FEED = 6

config = CollectorConfig(
    sleep_seconds=SLEEP_SECONDS,
    max_reviews_per_source=MAX_REVIEWS_PER_SOURCE,
    max_news_per_feed=MAX_NEWS_PER_FEED,
)

run_stores = stores.head(MAX_STORES).copy()
url_inventory = build_url_inventory(run_stores)

print(f"Running scrape for {len(run_stores)} stores")
print(f"URL inventory: {len(url_inventory)} URLs")
print("Google API keys: not used")
print("Azure OpenAI env keys: available for later enrichment if needed")
url_inventory.head(8)

Running scrape for 100 stores
URL inventory: 630 URLs
Google API keys: not used
Azure OpenAI env keys: available for later enrichment if needed


,store_id,retailer,city,state,source_type,source_name,url
0,USR-001,Walmart,New York,NY,reviews,sitejabber,https://www.sitejabber.com/reviews/walmart.com
1,USR-001,Walmart,New York,NY,reviews,trustpilot,https://www.trustpilot.com/review/www.walmart.com
2,USR-001,Walmart,New York,NY,reviews,google_news_reviews,https://news.google.com/rss/search?q=New+York+...
3,USR-001,Walmart,New York,NY,local_news,google_news_retail,https://news.google.com/rss/search?q=New+York+...
4,USR-001,Walmart,New York,NY,local_news,google_news_grocery,https://news.google.com/rss/search?q=New+York+...
5,USR-001,Walmart,New York,NY,local_news,google_news_local,https://news.google.com/rss/search?q=New+York+...
6,USR-001,Walmart,New York,NY,local_news,bing_news_retail,https://www.bing.com/news/search?q=New+York+NY...
7,USR-002,Target,Los Angeles,CA,reviews,sitejabber,https://www.sitejabber.com/reviews/target.com


## 4. Run multi-source scrape

For each store this fetches:
- **Reviews** from Sitejabber, Trustpilot, and Google News review roundups
- **News** from Google News RSS + Bing News RSS (4 feeds per store)

Expected runtime: ~2–4 min per store (~3–6 hours for 100 stores). Use `MAX_STORES=5` first.

In [17]:
reviews_df, news_df, log_df = collect_all_stores(run_stores, config)

print(f"Reviews collected: {len(reviews_df)}")
print(f"News collected:    {len(news_df)}")
print(f"Log rows:          {len(log_df)}")


[1/100] USR-001 | Walmart | New York, NY
    reviews | sitejabber | https://www.sitejabber.com/reviews/walmart.com...
    reviews | trustpilot | https://www.trustpilot.com/review/www.walmart.com...
    reviews | google_news_reviews | https://news.google.com/rss/search?q=New+York+NY+Walmart+customer+revi...
    news | google_news_retail | https://news.google.com/rss/search?q=New+York+NY+Walmart+retail+store&...
    news | google_news_grocery | https://news.google.com/rss/search?q=New+York+NY+grocery+shopping+reta...
    news | google_news_local | https://news.google.com/rss/search?q=New+York+NY+retail+opening+OR+sto...
    news | bing_news_retail | https://www.bing.com/news/search?q=New+York+NY+Walmart+retail&format=r...

[2/100] USR-002 | Target | Los Angeles, CA
    reviews | sitejabber | https://www.sitejabber.com/reviews/target.com...
    reviews | trustpilot | https://www.trustpilot.com/review/www.target.com...
    reviews | google_news_reviews | https://news.google.com/rss/search

## 5. Preview results

In [18]:
if not reviews_df.empty:
    display(reviews_df.groupby(["store_id", "source"]).size().unstack(fill_value=0).head(10))
    reviews_df.head(8)

source,google_news_reviews,sitejabber
store_id,,
USR-001,11,8
USR-002,8,8
USR-003,11,8
USR-004,8,8
USR-005,1,8
USR-006,9,8
USR-007,9,8
USR-008,8,8
USR-009,7,8


In [19]:
if not news_df.empty:
    display(news_df.groupby(["store_id", "event_type"]).size().unstack(fill_value=0).head(10))
    news_df.head(8)

event_type,construction,local_news,new_competitor,weather
store_id,,,,
USR-001,1,20,1,0
USR-002,1,19,1,0
USR-003,3,12,1,0
USR-004,2,11,2,1
USR-005,0,15,1,0
USR-006,0,16,2,0
USR-007,0,14,0,3
USR-008,0,16,2,0
USR-009,1,16,0,0


In [20]:
log_df.groupby(["source_type", "status"]).size().unstack(fill_value=0)

status,error,high,ok
source_type,,,
local_news,0,400,0
reviews,30,114,86


## 6. Save CSV + Excel

In [22]:
reviews_path = OUTPUT_DIR / "customer_reviews.csv"
news_path = OUTPUT_DIR / "local_news.csv"
log_path = OUTPUT_DIR / "scrape_run_log.csv"
excel_path = PROJECT_ROOT / "data" / "excel" / "scraped_100_reviews_news1.xlsx"
excel_path.parent.mkdir(parents=True, exist_ok=True)

reviews_df.to_csv(reviews_path, index=False)
news_df.to_csv(news_path, index=False)
log_df.to_csv(log_path, index=False)

with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
    run_stores.to_excel(writer, sheet_name="store_catalog", index=False)
    reviews_df.to_excel(writer, sheet_name="customer_reviews", index=False)
    news_df.to_excel(writer, sheet_name="local_news", index=False)
    log_df.to_excel(writer, sheet_name="scrape_run_log", index=False)

print("Saved:")
print(" ", reviews_path)
print(" ", news_path)
print(" ", log_path)
print(" ", excel_path)

Saved:
  d:\AICOE\Retail-StoreDNA\data\scraped_100\customer_reviews.csv
  d:\AICOE\Retail-StoreDNA\data\scraped_100\local_news.csv
  d:\AICOE\Retail-StoreDNA\data\scraped_100\scrape_run_log.csv
  d:\AICOE\Retail-StoreDNA\data\excel\scraped_100_reviews_news1.xlsx


## 7. Summary

In [ ]:
summary = pd.DataFrame({
    "metric": [
        "stores_scraped",
        "total_reviews",
        "total_news",
        "stores_with_reviews",
        "stores_with_news",
        "review_sources",
        "news_feeds",
    ],
    "value": [
        len(run_stores),
        len(reviews_df),
        len(news_df),
        reviews_df["store_id"].nunique() if not reviews_df.empty else 0,
        news_df["store_id"].nunique() if not news_df.empty else 0,
        reviews_df["source"].nunique() if not reviews_df.empty else 0,
        log_df[log_df["source_type"] == "local_news"]["source_name"].nunique() if not log_df.empty else 0,
    ],
})
summary